In [32]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [33]:
# Datasets AND DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0,1) => normalize(-1,1)
transform= transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

In [34]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [35]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [36]:
trainloader = DataLoader(trainset ,batch_size=64,shuffle=True)
testloader = DataLoader(testset ,batch_size=64)

# Build CNN

In [37]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size = 2 , stride = 2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) ,
            
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) ,
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10),

        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)
        return x

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

model = CNN().to(device)   # Move model to GPU

Using device: cuda


In [42]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN

In [43]:
epochs = 10

for epoch in range(epochs):
    training_loss = 0.0

    for images , labels in trainloader :

        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        output = model(images) # Forw. propag..
        loss=criterion(output,labels) # loss fnx

        loss.backward() # BP
        optimizer.step() # update params

        training_loss+= loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss = {training_loss/len(trainloader)}")

epoch=1/10 & loss = 1.3653376372269048
epoch=2/10 & loss = 0.9316955051001381
epoch=3/10 & loss = 0.7442036082448862
epoch=4/10 & loss = 0.612388584658008
epoch=5/10 & loss = 0.516318665250488
epoch=6/10 & loss = 0.4169513181118709
epoch=7/10 & loss = 0.33176134950707636
epoch=8/10 & loss = 0.25853658160742593
epoch=9/10 & loss = 0.19658943138483082
epoch=10/10 & loss = 0.16200053175230084


In [44]:
## Evcaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()
eval_loss= 0.0

with torch.no_grad():
    for images,labels in testloader:
        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {100 * correct_labels / total_labels:.2f}%")

Accuracy = 75.20%


In [ ]:
# Assignment :- calc loss for both train and eval and plot it on a chart